# Predicting extubation failure

Logistic regression and XGBoost predict recorded reintubation or death within 48 hours of planned extubation. Inputs are patient-level CSVs from the phenotyping notebook; this notebook handles splitting, preprocessing, modelling and evaluation.

## 1. Setup

Use Python 3.12. Install the pinned packages in `models/requirements.txt` once, then restart the kernel. From this notebook's `models` folder, run `%pip install -r requirements.txt` in a cell if needed.

In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import hashlib
import importlib.metadata
import json
import platform
import re
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
    confusion_matrix, log_loss, precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import DMatrix, XGBClassifier

from IPython.display import display, Image

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook within the group project folder containing data/.")

## 2. Settings

Edit `CONFIG`, then run all cells from top to bottom.

| Analysis | Cohort | Predictors | Purpose |
|---|---|---|---|
| `filtered_six_hour` | Supplied ≤50%-missingness cohort | Six hourly bins and static covariates | Main analysis |
| `full_six_hour` | Full eligible cohort before the missingness exclusion | Same predictors | Cohort sensitivity analysis |
| `filtered_last_hour` | Same patients as the main analysis | Bin 5 and the same static covariates | Feature ablation; set `include_last_hour_ablation=False` to omit |

The ablation uses last-hour summaries, which may already be forward-filled. These are not necessarily the last raw measurements. The full cohort retains the other phenotype exclusions, including unplanned extubations.

- `mode="check"`: synthetic end-to-end check, including final evaluation; no real patient CSV is read.
- `mode="trial"`: reduced tuning on both real development cohorts and the ablation; real holdouts stay closed.
- `mode="main"`: 5 outer/3 inner folds with the full search budget. Set `run_final_test=True` only after all analysis choices are fixed.

Outputs go to `models/outputs/<mode>/<analysis>/`. Combined tables are saved in the mode folder. The default is a real-data trial. LR tests six C values; XGBoost samples eight of the 48 listed combinations in main mode.

In [ ]:
CONFIG = {
    "execution": {
        "mode": "trial",            # check (synthetic), trial (reduced), or main
        "run_final_test": False,    # main mode only; enable after choices are frozen
    },
    "data_files": {
        "filtered": str(PROJECT_ROOT / "data/clean_final_dataset.csv"),
        "full": str(PROJECT_ROOT / "data/clean_full_final.csv"),
    },
    "metadata_csv": None,  # optional sidecar if temporal fields are absent from the CSVs
    "output_root": str(PROJECT_ROOT / "models/outputs"),
    "include_last_hour_ablation": True,
    "target": "combined_outcome",
    "seed": 42,
    "split_mode": "temporal",
    "temporal_cutoff": 2020,  # same calendar-year boundary for every analysis
    "test_fraction": 0.20,   # used only for an explicitly chosen stratified split
    "feature_columns": None,  # recognise physiological predictors; audit is saved
    "categorical_columns": ["gender", "sex", "first_careunit"],
    "rsbi_column": None,     # prefer last-hour RSBI
    "negative_followup_policy": "retain_recorded_labels",  # report uncertainty; do not redefine the cohort

    "outer_folds": 5,
    "inner_folds": 3,
    "lr_c_values": [0.001, 0.01, 0.1, 1, 10, 100],
    "lr_max_iter": 3000,
    "search_iterations": 8,
    "xgb_param_distributions": {
        "n_estimators": [200, 400],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.1],
        "min_child_weight": [1, 5],
        "reg_lambda": [1, 10],
    },
    "xgb_subsample": 0.8,
    "xgb_colsample_bytree": 0.8,
    "xgb_threads": 4,

    "calibrate": True,
    "calibration_method": "sigmoid",
    "sensitivity_target": 0.80,  # exploratory threshold target, not clinically validated
    "bootstrap_replicates": 1000,

    "trial_overrides": {
        "outer_folds": 2,
        "inner_folds": 2,
        "lr_c_values": [0.1, 1],
        "search_iterations": 1,
        "xgb_param_distributions": {
            "n_estimators": [30],
            "max_depth": [2, 3, 4],
            "learning_rate": [0.03, 0.1],
            "min_child_weight": [1, 5],
            "reg_lambda": [1, 10],
        },
        "xgb_threads": 1,
        "bootstrap_replicates": 20,  # exercises intervals in synthetic check mode
    },
}

## 3. Inputs and temporal split

Use `clean_final_dataset.csv` and `clean_full_final.csv` before median imputation or train/test splitting. Each patient must have one index extubation, consistent labels/features across the two files, and `extubation_time`, `anchor_year`, `anchor_year_group`. Set `metadata_csv` only when a sidecar is needed.

Predictors cover `[t−6 h, t)`, prediction is at planned extubation `t`, and the outcome window is `(t, t+48 h]`; there is no gap. The phenotyping code establishes eligibility, exclusions and pre-extubation availability. This notebook does not reconstruct the cohort or recompute its missingness filter.

Both cohorts use the explicit 2020 calendar-year boundary. Development patients have their entire possible year interval before 2020; test patients have their entire interval from 2020 onward. Overlapping intervals are excluded. Shared patients retain the same partition. Main and ablation also share the same outer folds.

`hospital_observed_48h` is checked as follow-up metadata, never a predictor. By default, supplied labels are retained and incomplete observation among development negatives is reported. Early discharge without a recorded event does not prove event-free 48-hour follow-up. `require_complete_negatives` stops on such labels for resolution by the phenotype definition; it does not silently exclude patients. Known positive events remain positive regardless of length of observation.

In [ ]:
REQUIRED_CONFIG = {
    "data_csv", "metadata_csv", "output_dir", "target", "seed", "split_mode",
    "temporal_cutoff", "test_fraction", "feature_columns", "categorical_columns",
    "rsbi_column", "outer_folds", "inner_folds", "lr_c_values", "lr_max_iter",
    "search_iterations", "xgb_param_distributions", "xgb_subsample",
    "xgb_colsample_bytree", "xgb_threads", "calibrate", "calibration_method",
    "sensitivity_target", "bootstrap_replicates", "synthetic_data", "run_mode",
    "analysis", "feature_set", "suite_output_dir", "negative_followup_policy", "pipeline_version",
}

PHYSIOLOGY = {
    "heart_rate", "mbp", "sbp", "dbp", "resp_rate", "temperature", "spo2",
    "respiratory_rate_total", "respiratory_rate_spontaneous",
    "tidal_volume_observed", "tidal_volume_spontaneous", "fio2", "peep",
    "plateau_pressure", "minute_volume", "minute_ventilation", "po2", "pco2",
    "ph", "fio2_bg", "fio2_chartevents", "pao2fio2ratio", "bicarbonate",
    "baseexcess", "lactate", "rsbi",
}
BASELINE = {"age", "admission_age", "gender", "sex", "first_careunit"}
OUTCOME_COMPONENTS = ["death_within_48_hours", "reintubation_within_48_hours"]
MODEL_NAMES = ["logistic_regression", "xgboost"]
PROBABILITY_MODELS = MODEL_NAMES + ["prevalence_baseline"]


def validate_config(config):
    cfg = deepcopy(config)
    missing = sorted(REQUIRED_CONFIG - set(cfg))
    if missing:
        raise ValueError("CONFIG is missing: " + str(missing))
    unknown = sorted(set(cfg) - REQUIRED_CONFIG)
    if unknown:
        raise ValueError("Unknown CONFIG settings: " + str(unknown))
    if cfg["run_mode"] not in {"main", "trial", "check"}:
        raise ValueError("CONFIG['execution']['mode'] must be 'check', 'trial' or 'main'.")
    if cfg["calibration_method"] not in {"sigmoid", "isotonic"}:
        raise ValueError("calibration_method must be 'sigmoid' or 'isotonic'.")
    if cfg["negative_followup_policy"] not in {"retain_recorded_labels", "require_complete_negatives"}:
        raise ValueError("Unknown negative_followup_policy.")
    if cfg["split_mode"] != "temporal" or cfg["temporal_cutoff"] is None:
        raise ValueError("These analyses require a common, explicit temporal cutoff.")
    return cfg


def make_run_configs(config):
    """Give every cohort/feature analysis its own folder within the selected mode."""
    cfg = deepcopy(config)
    execution = cfg.pop("execution", None)
    trial_overrides = cfg.pop("trial_overrides", None)
    files = cfg.pop("data_files")
    root = Path(cfg.pop("output_root"))
    include_ablation = cfg.pop("include_last_hour_ablation")
    if not isinstance(execution, dict) or set(execution) != {"mode", "run_final_test"}:
        raise ValueError("CONFIG['execution'] must define mode and run_final_test.")
    mode = execution["mode"]
    if mode not in {"main", "trial", "check"}:
        raise ValueError("CONFIG['execution']['mode'] must be 'check', 'trial' or 'main'.")
    if execution["run_final_test"] and mode == "trial":
        raise ValueError("Trial runs cannot evaluate the real holdout. Use main after freezing all analyses.")
    if mode in {"trial", "check"}:
        overrides = deepcopy(trial_overrides)
        allowed = {"outer_folds", "inner_folds", "search_iterations",
            "xgb_threads", "lr_c_values", "xgb_param_distributions", "bootstrap_replicates"}
        if not overrides or set(overrides) - allowed:
            raise ValueError("Trial overrides must specify reduced tuning settings, not change patients or the split.")
        cfg.update(overrides)
    cfg["synthetic_data"] = mode == "check"
    cfg["run_mode"] = mode
    cfg["pipeline_version"] = "two-cohort-1"
    cfg["suite_output_dir"] = str(root / mode)
    specs = [("filtered_six_hour", "filtered", "six_hour"),
             ("full_six_hour", "full", "six_hour")]
    if include_ablation:
        specs.append(("filtered_last_hour", "filtered", "last_hour"))
    return {name: validate_config({**deepcopy(cfg), "analysis": name,
        "feature_set": feature_set, "data_csv": files[cohort],
        "output_dir": str(root / mode / name)}) for name, cohort, feature_set in specs}


def write_json(path, value):
    """Save settings, converting NumPy scalars and paths to JSON values."""
    def encode(x):
        if isinstance(x, np.generic):
            return x.item()
        if isinstance(x, Path):
            return str(x)
        raise TypeError(f"Cannot encode {type(x)}")
    Path(path).write_text(json.dumps(value, indent=2, default=encode), encoding="utf-8")


def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def binary_series(s, name):
    """Require observed binary labels."""
    out = pd.to_numeric(s, errors="raise")
    if out.isna().any() or not out.isin([0, 1]).all():
        raise ValueError(f"{name} must contain only observed 0/1 labels.")
    return out.astype(int)


def forbidden_feature(name, target):
    n = name.lower()
    return (n == target.lower() or n in {"subject_id", "hadm_id", "stay_id", "dod", "label", "y"}
            or bool(re.search(r"outcome|death|deathtime|reintubat|discharg|dischtime|outtime|expire|los_|length_of_stay|followup|follow_up|hospital_observed|anchor_year|extubation_time|admittime|icu_intime|charttime|storetime|specimen_id", n)))


def recognised_feature(name):
    n = name.lower()
    if n in BASELINE:
        return True
    # Recognise hourly bins and whole-window summaries.
    base = re.sub(r"(_bin_[0-5]|_(mean|std|sd|min|max|last|slope|count|missing)(?:_6h)?|_6h)$", "", n)
    return base in PHYSIOLOGY

In [ ]:
def load_cohort(cfg):
    path = Path(cfg["data_csv"])
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path.resolve()}. Set data_csv to the full cohort before median imputation and splitting.")
    if path.stem.startswith(("train_set", "test_set", "X_train", "X_test")):
        raise ValueError("Use the full cohort before median imputation and splitting.")
    # Check duplicate headers before pandas renames them.
    with path.open(newline="", encoding="utf-8-sig") as f:
        header = next(csv.reader(f))
    if len(header) != len(set(header)):
        raise ValueError("Duplicate CSV column names; fix the export.")
    df = pd.read_csv(path, dtype={"subject_id": "string", "stay_id": "string", "hadm_id": "string"})
    if df.columns.duplicated().any():
        raise ValueError("Duplicate CSV column names; fix the export.")
    if "subject_id" not in df or cfg["target"] not in df:
        raise ValueError("Required columns: subject_id and " + cfg["target"])
    if df.subject_id.isna().any() or df.subject_id.duplicated().any():
        raise ValueError("Expected ONE row per subject_id, with a single index extubation.")
    # Keep splits reproducible if input rows are reordered.
    df = df.sort_values("subject_id").reset_index(drop=True)
    df[cfg["target"]] = binary_series(df[cfg["target"]], cfg["target"])
    components_present = [c in df for c in OUTCOME_COMPONENTS]
    if all(components_present):
        for c in OUTCOME_COMPONENTS:
            df[c] = binary_series(df[c], c)
        if not np.array_equal(df[cfg["target"]], df[OUTCOME_COMPONENTS].max(axis=1)):
            raise ValueError("combined_outcome disagrees with death OR reintubation.")
    else:
        warnings.warn("Outcome components are missing; cannot verify the combined label.")
    metadata_path = Path(cfg["metadata_csv"]) if cfg.get("metadata_csv") else None
    if metadata_path and metadata_path.exists():
        meta = pd.read_csv(metadata_path, dtype={"subject_id": "string"})
        if "subject_id" not in meta or meta.subject_id.isna().any() or meta.subject_id.duplicated().any():
            raise ValueError("Metadata must have one non-missing row per subject_id.")
        # Overlapping columns must agree before merging metadata.
        both = sorted((set(meta.columns) & set(df.columns)) - {"subject_id"})
        joined = df[["subject_id"] + both].merge(meta, on="subject_id", how="left", validate="one_to_one", suffixes=("_main", "_meta"))
        for c in both:
            a, b = joined[c + "_main"], joined[c + "_meta"]
            if not a.astype("string").fillna("<NA>").eq(b.astype("string").fillna("<NA>")).all():
                raise ValueError(f"Main CSV and metadata disagree in {c}; resolve before modelling.")
        extra = [c for c in meta if c not in df]
        df = df.merge(meta[["subject_id"] + extra], on="subject_id", how="left", validate="one_to_one")
    followup_col = next((c for c in ["hospital_observed_48h", "followup_complete"] if c in df), None)
    if followup_col:
        original = df[followup_col]
        known = pd.to_numeric(original.replace({True: 1, False: 0, "True": 1, "False": 0,
            "true": 1, "false": 0}), errors="coerce")
        if (original.notna() & (known.isna() | ~known.isin([0, 1]))).any():
            raise ValueError(f"{followup_col} must contain 0/1, booleans or missing values.")
        df[followup_col] = known
    else:
        known = pd.Series(np.nan, index=df.index)
    incomplete_negative = df[cfg["target"]].eq(0) & known.ne(1)
    if cfg["negative_followup_policy"] == "require_complete_negatives" and incomplete_negative.any():
        raise ValueError("Negative labels lack complete follow-up. Resolve the phenotype; no rows were silently dropped.")
    requested = cfg.get("feature_columns")
    selected = list(requested) if requested is not None else [c for c in df if recognised_feature(c)]
    if not selected or len(set(selected)) != len(selected):
        raise ValueError("No recognised predictors or duplicate features. Set feature_columns explicitly.")
    if not set(selected).issubset(df.columns):
        raise ValueError("Feature columns missing: " + str(sorted(set(selected) - set(df.columns))))
    leaks = [c for c in selected if forbidden_feature(c, cfg["target"])]
    if leaks:
        raise ValueError("Forbidden identifier/outcome/future predictors: " + str(leaks))
    cats = [c for c in selected if c in cfg["categorical_columns"]]
    nums = [c for c in selected if c not in cats]
    if not nums:
        raise ValueError("No numeric physiological predictors selected.")
    for c in nums:
        df[c] = pd.to_numeric(df[c], errors="raise").replace([np.inf, -np.inf], np.nan)
    for c in cats:
        df[c] = df[c].astype(object).where(df[c].notna(), np.nan)
    rsbi_candidates = [cfg["rsbi_column"]] if cfg.get("rsbi_column") else ["RSBI_bin_5", "rsbi_bin_5", "RSBI", "rsbi"]
    rsbi_col = next((c for c in rsbi_candidates if c in df), None)
    if cfg.get("rsbi_column") and rsbi_col is None:
        raise ValueError("Configured rsbi_column is absent.")
    if rsbi_col:
        df[rsbi_col] = pd.to_numeric(df[rsbi_col], errors="raise").replace([np.inf, -np.inf], np.nan)
        # Non-positive RSBI values are invalid measurements.
        df.loc[df[rsbi_col] <= 0, rsbi_col] = np.nan
    else:
        warnings.warn("No RSBI column; skipping the RSBI comparison.")
    audit = pd.DataFrame({"column": df.columns})
    audit["used_as_predictor"] = audit.column.isin(selected)
    audit["reason"] = ["predictor" if c in selected else "blocked ID/outcome/time/follow-up" if forbidden_feature(c, cfg["target"]) else "metadata or unrecognised: not selected" for c in audit.column]
    hashes = {"data_csv_sha256": file_hash(path)}
    if metadata_path and metadata_path.exists():
        hashes["metadata_csv_sha256"] = file_hash(metadata_path)
    return df, nums, cats, rsbi_col, audit, hashes


def followup_summary(df, cfg):
    col = next((c for c in ["hospital_observed_48h", "followup_complete"] if c in df), None)
    observed = df[col].eq(1) if col else pd.Series(False, index=df.index)
    return {"followup_field": col, "patients": len(df),
        "negative_labels_without_complete_observation": int((df[cfg["target"]].eq(0) & ~observed).sum()),
        "known_events_without_complete_observation": int((df[cfg["target"]].eq(1) & ~observed).sum()),
        "policy": cfg["negative_followup_policy"]}

In [ ]:
def make_split(df, cfg):
    manifest = df[["subject_id"]].copy()
    manifest["partition"] = "ambiguous_time"
    if cfg["split_mode"] == "temporal":
        required = ["extubation_time", "anchor_year", "anchor_year_group"]
        missing = [c for c in required if c not in df]
        if missing:
            raise ValueError("Temporal holdout needs " + str(missing) + ". Include them in data_csv or metadata_csv, keyed by subject_id.")
        groups = df.anchor_year_group.astype(str).str.extract(r"^\s*(\d{4})\s*[-–]\s*(\d{4})\s*$").astype(float)
        anchor = pd.to_numeric(df.anchor_year, errors="coerce")
        year = pd.to_datetime(df.extubation_time, errors="coerce").dt.year
        if groups.isna().any().any() or anchor.isna().any() or year.isna().any():
            raise ValueError("Temporal metadata contain missing or invalid values.")
        if (groups[1] < groups[0]).any() or (anchor % 1 != 0).any():
            raise ValueError("Invalid anchor year or year-group interval.")
        offset = year - anchor
        # Map shifted extubation years to possible calendar-year intervals.
        manifest["possible_year_lower"] = (groups[0] + offset).astype(int)
        manifest["possible_year_upper"] = (groups[1] + offset).astype(int)
        cutoff = cfg["temporal_cutoff"] if cfg["temporal_cutoff"] is not None else int(groups[0].max())
        manifest.loc[manifest.possible_year_upper < cutoff, "partition"] = "development"
        manifest.loc[manifest.possible_year_lower >= cutoff, "partition"] = "test"
        split_note = {"mode": "temporal", "cutoff": int(cutoff), "rule": "development: upper possible year < cutoff; test: lower possible year >= cutoff; overlapping intervals excluded", "cutoff_choice": "explicit" if cfg["temporal_cutoff"] is not None else "start of latest observed anchor_year_group; chosen without model performance"}
    elif cfg["split_mode"] == "stratified":
        dev, test = train_test_split(np.arange(len(df)), test_size=cfg["test_fraction"], stratify=df[cfg["target"]], random_state=cfg["seed"])
        manifest.loc[dev, "partition"] = "development"
        manifest.loc[test, "partition"] = "test"
        split_note = {"mode": "stratified", "deviation": "Random patient holdout, NOT temporal validation", "test_fraction": cfg["test_fraction"]}
    else:
        raise ValueError("split_mode must be temporal or stratified.")
    dev = np.flatnonzero(manifest.partition.eq("development"))
    test = np.flatnonzero(manifest.partition.eq("test"))
    if not len(dev) or not len(test):
        raise ValueError("Empty development/test set. Check year intervals and temporal_cutoff.")
    assert not set(df.iloc[dev].subject_id) & set(df.iloc[test].subject_id)
    # Validate fold sizes using development labels only.
    checked_cv(df.iloc[dev][cfg["target"]].to_numpy(), cfg["outer_folds"], cfg["seed"])
    return manifest, dev, test, split_note


def checked_cv(y, folds, seed):
    counts = np.bincount(np.asarray(y, dtype=int), minlength=2)
    if folds < 2 or min(counts) < folds:
        raise ValueError(f"Need at least {folds} patients in each class for cross-validation; found {counts.tolist()}.")
    return list(StratifiedKFold(folds, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))


def check_inputs(config):
    """Check inputs and partition sizes without training or scoring test patients."""
    cfg = validate_config(config)
    df, nums, cats, rsbi_col, _, _ = load_cohort(cfg)
    manifest, dev, test, _ = make_split(df, cfg)
    y = df.iloc[dev][cfg["target"]].to_numpy()
    for fold, (train, _) in enumerate(checked_cv(y, cfg["outer_folds"], cfg["seed"])):
        checked_cv(y[train], cfg["inner_folds"], cfg["seed"] + fold + 1)
    return {"mode": cfg["run_mode"], "patients": len(df), "development_patients": len(dev),
        "development_events": int(y.sum()), "reserved_test_patients": len(test),
        "excluded_ambiguous_year": int(manifest.partition.eq("ambiguous_time").sum()),
        "predictors": len(nums) + len(cats), "rsbi_comparator": rsbi_col,
        "outer_folds": cfg["outer_folds"], "inner_folds": cfg["inner_folds"],
        "calibration": cfg["calibration_method"] if cfg["calibrate"] else "disabled",
        "development_negatives_without_complete_followup": followup_summary(df.iloc[dev], cfg)["negative_labels_without_complete_observation"]}


def prepare_analyses(configs):
    """Lock shared predictors and check that cohorts represent the same index events."""
    configs = deepcopy(configs)
    main = configs["filtered_six_hour"]
    filtered, nums, cats, _, _, _ = load_cohort(main)
    features = nums + cats
    full, full_nums, full_cats, _, _, _ = load_cohort(configs["full_six_hour"])
    if set(features) != set(full_nums + full_cats):
        raise ValueError("Filtered and full cohorts must have the same predictor definitions.")
    if not set(filtered.subject_id).issubset(set(full.subject_id)):
        raise ValueError("The filtered cohort must be a subset of the full eligible cohort.")
    shared = full.set_index("subject_id").loc[filtered.subject_id].reset_index()
    check_columns = features + [main["target"], "extubation_time", "anchor_year", "anchor_year_group"]
    check_columns += [c for c in ["stay_id", "hadm_id", *OUTCOME_COMPONENTS,
        "hospital_observed_48h", "followup_complete"] if c in filtered or c in full]
    try:
        pd.testing.assert_frame_equal(filtered[check_columns].reset_index(drop=True),
            shared[check_columns].reset_index(drop=True), check_dtype=False,
            check_exact=False, rtol=1e-8, atol=1e-10)
    except (AssertionError, KeyError) as exc:
        raise ValueError("Shared patients have inconsistent features, outcomes or index-event metadata across the two CSVs.") from None
    for cfg in configs.values():
        cfg["feature_columns"] = features.copy()
    if "filtered_last_hour" in configs:
        physiological = [c for c in features if c.lower() not in BASELINE]
        if any(not re.search(r"_bin_[0-5]$", c, flags=re.I) for c in physiological):
            raise ValueError("Last-hour ablation requires hourly-bin physiology. Set feature_columns to the hourly features or disable the ablation.")
        last = [c for c in features if c.lower() in BASELINE or c.lower().endswith("_bin_5")]
        families = {re.sub(r"_bin_[0-5]$", "", c.lower()) for c in physiological}
        last_families = {c.lower()[:-6] for c in last if c.lower().endswith("_bin_5")}
        if families != last_families:
            raise ValueError("Every physiological family must have a bin_5 column for a matched ablation.")
        configs["filtered_last_hour"]["feature_columns"] = last
    main_manifest, _, _, _ = make_split(filtered, main)
    full_manifest, _, _, _ = make_split(full, configs["full_six_hour"])
    matching = full_manifest.set_index("subject_id").loc[filtered.subject_id, "partition"].to_numpy()
    if not np.array_equal(main_manifest.partition.to_numpy(), matching):
        raise ValueError("Shared patients must have the same temporal partition in both cohorts.")
    return configs


def environment_versions():
    versions = {p: importlib.metadata.version(p) for p in
        ["numpy", "pandas", "scikit-learn", "xgboost", "scipy", "matplotlib", "joblib"]}
    return {**versions, "python": platform.python_version()}


def suite_plan(configs):
    return {"analyses": {name: {"config": cfg, "source_hashes": load_cohort(cfg)[-1]}
        for name, cfg in configs.items()}, "environment": environment_versions()}


def guard_final_suite(configs):
    """Once any holdout is evaluated, all analyses must keep their frozen inputs/models."""
    marker = Path(next(iter(configs.values()))["suite_output_dir"]) / "final_suite_started.json"
    if not marker.exists():
        return
    saved = json.loads(marker.read_text())
    if saved["plan"] != suite_plan(configs):
        raise RuntimeError("Final evaluation has started. Restore the frozen analysis set, settings, data and environment.")
    for name, cfg in configs.items():
        out = Path(cfg["output_dir"])
        if file_hash(out / "trained_models.joblib") != saved["model_hashes"][name]:
            raise RuntimeError("A model changed after final evaluation started.")
        if file_hash(out / "frozen_analysis.json") != saved["analysis_hashes"][name]:
            raise RuntimeError("Frozen analysis metadata changed after final evaluation started.")

In [ ]:
def prepare_synthetic_check(configs):
    """Create a fixed software-test fixture; never read the patient CSVs."""
    cfg = configs["filtered_six_hour"]
    if cfg["run_mode"] != "check" or not cfg["synthetic_data"]:
        raise ValueError("Synthetic fixtures are only available in check mode.")
    folder = Path(cfg["suite_output_dir"]) / "synthetic_inputs"
    folder.mkdir(parents=True, exist_ok=True)
    n = 360
    rng = np.random.default_rng(2026)
    severity = rng.normal(size=n)
    y = (rng.random(n) < 1 / (1 + np.exp(1.4 - .8 * severity))).astype(int)
    death = (y & (rng.random(n) < .3)).astype(int)
    df = pd.DataFrame({
        "subject_id": np.arange(10000000, 10000000 + n).astype(str),
        "combined_outcome": y, "death_within_48_hours": death,
        "reintubation_within_48_hours": y, "hospital_observed_48h": True,
        "age": rng.integers(20, 90, n), "gender": rng.choice(["F", "M"], n),
        "hospital_expire_flag": death,
    })
    for hour in range(6):
        df[f"heart_rate_bin_{hour}"] = 80 + 8 * severity + rng.normal(0, 6, n)
        df[f"resp_rate_bin_{hour}"] = 18 + 2 * severity + rng.normal(0, 2, n)
        df.loc[rng.random(n) < .15, f"heart_rate_bin_{hour}"] = np.nan
        df[f"lactate_bin_{hour}"] = np.nan  # exercises entirely missing predictors
        df[f"RSBI_bin_{hour}"] = np.clip(70 + 15 * severity + rng.normal(0, 10, n), 1, None)
        df.loc[rng.random(n) < .3, f"RSBI_bin_{hour}"] = np.nan
    df.loc[:9, "hospital_observed_48h"] = False
    cutoff = cfg["temporal_cutoff"]
    groups = np.repeat(f"{cutoff-9} - {cutoff-7}", n).astype(object)
    groups[-n // 4:] = f"{cutoff} - {cutoff+2}"
    groups[:12] = f"{cutoff-3} - {cutoff-1}"
    meta = pd.DataFrame({"subject_id": df.subject_id, "anchor_year": 2140,
        "anchor_year_group": groups, "extubation_time": "2140-06-15 14:00:00"})
    # These shifted intervals cross the cutoff and must be excluded.
    meta.loc[:11, "extubation_time"] = "2141-06-15 14:00:00"
    df = df.merge(meta, on="subject_id", validate="one_to_one")
    df.to_csv(folder / "full.csv", index=False)
    df.loc[np.arange(n) % 5 != 0].to_csv(folder / "filtered.csv", index=False)
    return {name: {**c, "data_csv": str(folder / ("full.csv" if name == "full_six_hour" else "filtered.csv")),
        "metadata_csv": None} for name, c in configs.items()}

In [ ]:
RUN_CONFIGS = make_run_configs(CONFIG)
if CONFIG["execution"]["mode"] == "check":
    RUN_CONFIGS = prepare_synthetic_check(RUN_CONFIGS)
    print("SYNTHETIC SOFTWARE CHECK — not research results.")
RUN_CONFIGS = prepare_analyses(RUN_CONFIGS)
guard_final_suite(RUN_CONFIGS)
INPUT_CHECKS = pd.DataFrame({name: check_inputs(cfg) for name, cfg in RUN_CONFIGS.items()}).T
display(INPUT_CHECKS)

## 4. Models, tuning and calibration

LR models linear associations; XGBoost permits nonlinear effects and interactions. Inner CV tunes by AUROC; outer folds evaluate the tuned, calibrated pipelines. Median imputation, missingness indicators, LR scaling and class weights are learned within training folds. LR uses balanced class weights; XGBoost uses the training-fold negative/positive ratio. Sigmoid calibration uses cross-validated predictions within each outer training partition.

In [ ]:
class FoldBalancedXGB(ClassifierMixin, BaseEstimator):
    """Calculate class weights from the current training fold."""
    def __init__(self, n_estimators=250, max_depth=3, learning_rate=0.05,
                 min_child_weight=1, reg_lambda=1, subsample=0.8,
                 colsample_bytree=0.8, random_state=42, n_jobs=4):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.min_child_weight = min_child_weight
        self.reg_lambda = reg_lambda
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.random_state = random_state
        self.n_jobs = n_jobs

    def fit(self, X, y):
        counts = np.bincount(np.asarray(y, dtype=int), minlength=2)
        if min(counts) == 0:
            raise ValueError("XGBoost fit requires both classes.")
        self.scale_pos_weight_ = float(counts[0] / counts[1])
        self.model_ = XGBClassifier(**self.get_params(), scale_pos_weight=self.scale_pos_weight_,
            objective="binary:logistic", eval_metric="logloss", tree_method="hist", device="cpu")
        self.model_.fit(X, y)
        self.classes_ = self.model_.classes_
        self.n_features_in_ = self.model_.n_features_in_
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    def predict(self, X):
        return self.model_.predict(X)


def model_pipeline(name, nums, cats, cfg):
    numeric = [("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True))]
    if name == "logistic_regression":
        numeric.append(("scale", StandardScaler()))
    parts = [("numeric", Pipeline(numeric), nums)]
    if cats:
        categorical = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown", keep_empty_features=True)),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
        parts.append(("categorical", categorical, cats))
    prep = ColumnTransformer(parts, remainder="drop", sparse_threshold=0)
    if name == "logistic_regression":
        model = LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=cfg["lr_max_iter"], random_state=cfg["seed"])
    elif name == "xgboost":
        model = FoldBalancedXGB(random_state=cfg["seed"], n_jobs=cfg["xgb_threads"],
            subsample=cfg["xgb_subsample"], colsample_bytree=cfg["xgb_colsample_bytree"])
    else:
        raise ValueError(name)
    return Pipeline([("preprocess", prep), ("model", model)])


def fit_tuned(name, X, y, nums, cats, cfg, cv):
    pipe = model_pipeline(name, nums, cats, cfg)
    common = dict(scoring={"auroc": "roc_auc", "average_precision": "average_precision"},
        refit="auroc", cv=cv, n_jobs=1, error_score="raise", return_train_score=False)
    if name == "logistic_regression":
        search = GridSearchCV(pipe, {"model__C": cfg["lr_c_values"]}, **common)
    else:
        params = {"model__" + key: values for key, values in cfg["xgb_param_distributions"].items()}
        search = RandomizedSearchCV(pipe, params, n_iter=cfg["search_iterations"], random_state=cfg["seed"], **common)
    search.fit(X, y)
    fitted = search.best_estimator_
    if cfg["calibrate"]:
        # Calibrate within training folds, then refit on this training partition.
        fitted = CalibratedClassifierCV(clone(fitted), method=cfg["calibration_method"], cv=cv, ensemble=False, n_jobs=1)
        fitted.fit(X, y)
    cols = [c for c in search.cv_results_ if c == "params" or c.startswith(("mean_test_", "std_test_", "rank_test_", "mean_fit_"))]
    return fitted, search.best_params_, pd.DataFrame(search.cv_results_)[cols]

## 5. Metrics and thresholds

Report AUROC, average precision (the step-based PR summary), Brier score, log loss, sensitivity, specificity, PPV and NPV. Prevalence and fixed-seed random predictions are reference baselines. RSBI uses threshold 105 and is compared with both models on the same RSBI-available patients.

For each fitted pipeline, choose the development OOF threshold giving maximum specificity while reaching at least 80% sensitivity. This exploratory target prioritises detecting failures while showing the resulting false-alarm burden; it is not a clinically validated operating point. Threshold-selection metrics are descriptive. Freeze each threshold before the test evaluation.

In [ ]:
def safe_ratio(a, b):
    return float(a / b) if b else np.nan


def metrics(y, score, threshold=0.5, probability=True):
    """Compute classification metrics and, for probabilities, calibration losses."""
    y, score = np.asarray(y, int), np.asarray(score, float)
    keep = np.isfinite(score)
    y, score = y[keep], score[keep]
    if not len(y):
        return {"n": 0}
    two = len(np.unique(y)) == 2
    tn, fp, fn, tp = confusion_matrix(y, score >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "events": int(y.sum()), "prevalence": float(y.mean()),
        "auroc": roc_auc_score(y, score) if two else np.nan,
        "average_precision": average_precision_score(y, score) if two else np.nan,
        "threshold": float(threshold), "sensitivity": safe_ratio(tp, tp + fn),
        "specificity": safe_ratio(tn, tn + fp), "ppv": safe_ratio(tp, tp + fp),
        "npv": safe_ratio(tn, tn + fn), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    result["brier"] = brier_score_loss(y, score) if probability else np.nan
    result["log_loss"] = log_loss(y, np.clip(score, 1e-7, 1 - 1e-7), labels=[0, 1]) if probability else np.nan
    return result


def choose_threshold(y, score, sensitivity_target):
    """Maximise specificity at the target sensitivity using development OOF scores."""
    if not 0 < sensitivity_target <= 1:
        raise ValueError("sensitivity_target must be in (0,1].")
    fpr, tpr, thresholds = roc_curve(y, score, drop_intermediate=False)
    eligible = np.flatnonzero((tpr >= sensitivity_target) & np.isfinite(thresholds))
    # roc_curve thresholds descend: among tied FPR choose the highest threshold.
    best = eligible[np.argmin(fpr[eligible])]
    return float(thresholds[best])


def prediction_table(df, cfg, scores, rsbi_col, fold=None):
    cols = [c for c in ["subject_id", "hadm_id", "stay_id", cfg["target"], *OUTCOME_COMPONENTS, "age", "admission_age", "gender", "sex"] if c in df]
    out = df[cols].copy().reset_index(drop=True)
    out = out.rename(columns={cfg["target"]: "y_true"})
    for name, values in scores.items():
        out[name] = values
    if rsbi_col:
        out["rsbi"] = df[rsbi_col].to_numpy()
    if fold is not None:
        out["outer_fold"] = fold
    return out


def comparison_tables(predictions, thresholds):
    rows = []
    scopes = {"all_patients": np.ones(len(predictions), dtype=bool)}
    if "rsbi" in predictions:
        scopes["rsbi_available_same_patients"] = np.isfinite(predictions.rsbi)
    for scope, mask in scopes.items():
        p = predictions.loc[mask]
        for name in PROBABILITY_MODELS + ["random_guess"]:
            if name in p:
                # Calibration losses apply to probability models only.
                rows.append({"scope": scope, "model": name, **metrics(p.y_true, p[name], thresholds.get(name, 0.5), name in PROBABILITY_MODELS)})
        if scope == "rsbi_available_same_patients":
            rows.append({"scope": scope, "model": "rsbi", **metrics(p.y_true, p.rsbi, 105, False)})
    return pd.DataFrame(rows)


def plot_comparison(p, folder, prefix, include_rsbi=False):
    folder = Path(folder)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    calibration_points = []
    if p.y_true.nunique() == 2:
        names = PROBABILITY_MODELS + ["random_guess"] + (["rsbi"] if include_rsbi else [])
        for name in names:
            fpr, tpr, _ = roc_curve(p.y_true, p[name])
            precision, recall, _ = precision_recall_curve(p.y_true, p[name])
            axes[0].plot(fpr, tpr, label=name)
            # Step interpolation matches average precision.
            axes[1].step(recall, precision, where="post", label=name)
        axes[0].plot([0, 1], [0, 1], "k--", alpha=.5)
        axes[1].axhline(p.y_true.mean(), color="k", linestyle="--", label="prevalence")
        for name in PROBABILITY_MODELS:
            observed, predicted = calibration_curve(p.y_true, p[name], n_bins=8, strategy="quantile")
            axes[2].plot(predicted, observed, "o-", label=name)
            calibration_points.extend({"model": name, "mean_predicted_probability": x,
                "observed_event_rate": y} for x, y in zip(predicted, observed))
    axes[2].plot([0, 1], [0, 1], "k--", alpha=.5)
    for ax, xlabel, ylabel in zip(axes, ["False positive rate", "Recall", "Mean predicted probability"], ["Sensitivity", "Precision", "Observed event rate"]):
        ax.set(xlabel=xlabel, ylabel=ylabel, xlim=(0, 1), ylim=(0, 1))
        ax.legend(fontsize=7)
    if calibration_points:
        points = pd.DataFrame(calibration_points)
        points.to_csv(folder / f"{prefix}_calibration_points.csv", index=False)
        limit = min(1, max(.1, float(points[["mean_predicted_probability", "observed_event_rate"]].max().max()) * 1.15))
        axes[2].set(xlim=(0, limit), ylim=(0, limit))
    fig.suptitle(prefix + ": retrospective predictions")
    fig.tight_layout()
    fig.savefig(folder / f"{prefix}_curves.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

## 6. Development and comparison

Run all prespecified analyses before inspecting any holdout. Mean outer-fold AUROC selects LR or XGBoost separately within each analysis; the filtered six-hour analysis is primary. Both models are then retuned/refitted on that analysis's complete development cohort. Models, features, thresholds, source hashes and environment versions are saved. Completed development is reused when settings, data and environment match, so enabling final evaluation does not repeat fitting.

`development_analysis_comparison.csv` reports all analyses together. Mean fold AUROC and pooled OOF AUROC are distinct summaries. Filtered/full results combine population and model changes; their difference is not a paired estimate of the effect of filtering. The feature ablation keeps the filtered patients and folds fixed.

In [ ]:
def run_development(config):
    cfg = validate_config(config)
    out = Path(cfg["output_dir"])
    df, nums, cats, rsbi_col, audit, hashes = load_cohort(cfg)
    if (out / "frozen_analysis.json").exists():
        saved_mode = json.loads((out / "frozen_analysis.json").read_text())["config"].get("run_mode", "main")
        if saved_mode != cfg["run_mode"]:
            raise ValueError("This output directory belongs to a different run mode; use separate output folders for each mode.")
    locked = (out / "final_test_started.json").exists() or (Path(cfg["suite_output_dir"]) / "final_suite_started.json").exists()
    cache_files = ["frozen_analysis.json", "trained_models.joblib", "development_oof_predictions.csv",
        "patient_split_manifest.csv", "environment_versions.json", "threshold_derivation_DESCRIPTIVE_NOT_VALIDATION.csv"]
    cached = all((out / name).exists() for name in cache_files)
    reuse = False
    if cached or locked:
        frozen = json.loads((out / "frozen_analysis.json").read_text())
        reuse = (cached and cfg == frozen["config"] and hashes == frozen["source_hashes"]
            and json.loads((out / "environment_versions.json").read_text()) == environment_versions())
        if locked and not reuse:
            raise RuntimeError("Final test already opened and settings/data/environment changed. Restore the frozen analysis.")
    if reuse:
        bundle = joblib.load(out / "trained_models.joblib")
        dev = df.set_index("subject_id", drop=False).loc[bundle["development_ids"]].reset_index(drop=True)
        print("Reusing completed development: settings, source data and environment are unchanged.")
        return {"bundle": bundle, "data": df, "development": dev,
            "oof": pd.read_csv(out / "development_oof_predictions.csv", dtype={"subject_id": "string"}),
            "manifest": pd.read_csv(out / "patient_split_manifest.csv", dtype={"subject_id": "string"}), "output_dir": out}
    manifest, dev_idx, test_idx, split_note = make_split(df, cfg)
    out.mkdir(parents=True, exist_ok=True)
    dev = df.iloc[dev_idx].reset_index(drop=True)
    followup = followup_summary(dev, cfg)
    write_json(out / "development_followup.json", followup)
    if followup["negative_labels_without_complete_observation"]:
        warnings.warn(f"{cfg['analysis']}: {followup['negative_labels_without_complete_observation']} development negatives lack complete 48-hour observation. Supplied labels are retained; describe this limitation.")
    X, y = dev[nums + cats], dev[cfg["target"]].to_numpy()
    if min(np.bincount(y, minlength=2)) < 50:
        warnings.warn("Fewer than 50 development patients in one class; estimates may be unstable.")
    outer = checked_cv(y, cfg["outer_folds"], cfg["seed"])
    scores = {n: np.full(len(dev), np.nan) for n in PROBABILITY_MODELS}
    # Fixed random scores provide a reproducible chance baseline.
    scores["random_guess"] = np.random.default_rng(cfg["seed"]).uniform(size=len(dev))
    folds = np.full(len(dev), -1, dtype=int)
    fold_results, best_params = [], []
    for fold, (train, valid) in enumerate(outer):
        inner = checked_cv(y[train], cfg["inner_folds"], cfg["seed"] + fold + 1)
        folds[valid] = fold
        scores["prevalence_baseline"][valid] = y[train].mean()
        for name in MODEL_NAMES:
            print(f"Outer fold {fold + 1}/{len(outer)}: {name}", flush=True)
            fitted, params, search_table = fit_tuned(name, X.iloc[train], y[train], nums, cats, cfg, inner)
            scores[name][valid] = fitted.predict_proba(X.iloc[valid])[:, 1]
            search_table.to_csv(out / f"search_outer_{fold}_{name}.csv", index=False)
            best_params.append({"outer_fold": fold, "model": name, "parameters": params})
        for name, values in scores.items():
            fold_results.append({"outer_fold": fold, "model": name, **metrics(y[valid], values[valid], 0.5, name in PROBABILITY_MODELS)})
    assert all(np.isfinite(s).all() for s in scores.values()) and (folds >= 0).all()
    oof = prediction_table(dev, cfg, scores, rsbi_col, folds)
    oof.to_csv(out / "development_oof_predictions.csv", index=False)
    manifest["outer_fold"] = -1
    manifest.loc[dev_idx, "outer_fold"] = folds
    manifest.to_csv(out / "patient_split_manifest.csv", index=False)
    pd.DataFrame(fold_results).to_csv(out / "outer_fold_metrics_fixed_0p5.csv", index=False)
    comparison_tables(oof, {}).to_csv(out / "development_metrics_fixed_0p5.csv", index=False)
    fold_frame = pd.DataFrame(fold_results)
    fold_frame.groupby("model")[["auroc", "average_precision", "brier", "sensitivity", "specificity"]].agg(["mean", "std"]).to_csv(out / "outer_fold_summary_mean_sd.csv")
    ranked = fold_frame[fold_frame.model.isin(MODEL_NAMES)].groupby("model").auroc.mean().sort_values(ascending=False)
    winner = str(ranked.index[0])
    thresholds = {name: choose_threshold(y, scores[name], cfg["sensitivity_target"]) for name in MODEL_NAMES}
    thresholds.update({"prevalence_baseline": 0.5, "random_guess": 0.5})
    # Threshold-selection scores are descriptive, not independent validation.
    comparison_tables(oof, thresholds).to_csv(out / "threshold_derivation_DESCRIPTIVE_NOT_VALIDATION.csv", index=False)
    plot_comparison(oof, out, "development_oof")
    if "rsbi" in oof:
        plot_comparison(oof.loc[np.isfinite(oof.rsbi)], out, "development_rsbi_matched_subset", include_rsbi=True)
    final_models = {}
    final_cv = checked_cv(y, cfg["inner_folds"], cfg["seed"] + 100)
    final_params = {}
    for name in MODEL_NAMES:
        print(f"Refitting {name} on development patients only", flush=True)
        fitted, params, search_table = fit_tuned(name, X, y, nums, cats, cfg, final_cv)
        final_models[name] = fitted
        final_params[name] = params
        search_table.to_csv(out / f"search_final_development_{name}.csv", index=False)
    # Describe missingness on development patients only.
    dev[nums + cats].isna().mean().rename("missing_fraction_development").to_csv(out / "development_feature_missingness.csv")
    audit.to_csv(out / "feature_audit.csv", index=False)
    write_json(out / "outer_best_parameters.json", best_params)
    versions = environment_versions()
    write_json(out / "environment_versions.json", versions)
    frozen = {"config": cfg, "source_hashes": hashes, "split": split_note,
        "selected_features": nums + cats, "rsbi_column": rsbi_col,
        "partition_counts": manifest.partition.value_counts().to_dict(),
        "thresholds": thresholds, "threshold_rule": "maximum specificity at development OOF sensitivity >= configured target; exploratory, not clinically validated",
        "selected_primary_model": winner, "selection_rule": "highest mean outer-fold AUROC, before final test",
        "mean_outer_fold_auroc": ranked.to_dict(), "final_parameters": final_params,
        "development_n": len(dev), "development_events": int(y.sum()),
        "development_followup": followup,
        "test_outcomes_used_for_training_or_selection": False}
    write_json(out / "frozen_analysis.json", frozen)
    bundle = {"models": final_models, "frozen": frozen, "development_prevalence": float(y.mean()),
        "development_ids": dev.subject_id.tolist(), "test_ids": df.iloc[test_idx].subject_id.tolist()}
    joblib.dump(bundle, out / "trained_models.joblib")
    print(f"{cfg['run_mode'].capitalize()} development complete. Selected model: {winner}. Final test not evaluated.")
    return {"bundle": bundle, "data": df, "development": dev, "oof": oof, "manifest": manifest, "output_dir": out}


def export_analysis_comparison(runs, final=False):
    """Keep population changes separate from the matched feature comparison."""
    rows = []
    for name, run in runs.items():
        frozen = run["bundle"]["frozen"]
        file = "FINAL_test_metrics.csv" if final else "threshold_derivation_DESCRIPTIVE_NOT_VALIDATION.csv"
        table = pd.read_csv(run["output_dir"] / file)
        table = table.loc[table.scope.eq("all_patients")].copy()
        table.insert(0, "analysis", name)
        table["feature_set"] = frozen["config"]["feature_set"]
        table["predictors"] = len(frozen["selected_features"])
        table["selected_primary_model"] = frozen["selected_primary_model"]
        table["development_patients"] = frozen["development_n"]
        table["development_prevalence"] = frozen["development_events"] / frozen["development_n"]
        table["reserved_test_patients"] = frozen["partition_counts"].get("test", 0)
        table["excluded_ambiguous_year"] = frozen["partition_counts"].get("ambiguous_time", 0)
        table["temporal_cutoff"] = frozen["split"]["cutoff"]
        table["mean_outer_fold_auroc"] = table.model.map(frozen["mean_outer_fold_auroc"])
        table["evaluation"] = "frozen temporal holdout" if final else "pooled development OOF; threshold metrics descriptive"
        rows.append(table)
    result = pd.concat(rows, ignore_index=True)
    folder = Path(next(iter(runs.values()))["bundle"]["frozen"]["config"]["suite_output_dir"])
    result.to_csv(folder / ("FINAL_analysis_comparison.csv" if final else "development_analysis_comparison.csv"), index=False)
    return result


def verify_ablation_pair(runs):
    if "filtered_last_hour" not in runs:
        return
    main, ablation = runs["filtered_six_hour"], runs["filtered_last_hour"]
    for field in ["development_ids", "test_ids"]:
        if main["bundle"][field] != ablation["bundle"][field]:
            raise ValueError("The feature ablation must use exactly the same patients and temporal split.")
    pd.testing.assert_frame_equal(main["oof"][["subject_id", "y_true", "outer_fold"]],
        ablation["oof"][["subject_id", "y_true", "outer_fold"]])

In [ ]:
guard_final_suite(RUN_CONFIGS)
runs = {}
for analysis, cfg in RUN_CONFIGS.items():
    print(f"\n{analysis} ({cfg['run_mode']})", flush=True)
    runs[analysis] = run_development(cfg)
verify_ablation_pair(runs)
development_comparison = export_analysis_comparison(runs)
display(development_comparison[["analysis", "model", "n", "prevalence", "mean_outer_fold_auroc",
    "auroc", "average_precision", "brier", "sensitivity", "specificity", "ppv"]])
print("AUROC/AP columns are pooled OOF; mean_outer_fold_auroc selects the model. Threshold metrics are descriptive.")
for analysis, run in runs.items():
    print(analysis, "selected model:", run["bundle"]["frozen"]["selected_primary_model"])
    display(Image(filename=str(run["output_dir"] / "development_oof_curves.png")))

## 7. Interpretation

LR coefficients and TreeSHAP describe the fitted base models before calibration. They show predictive associations, not causal effects. TreeSHAP values are in base-model log odds and use development patients only.

In [ ]:
def base_pipeline(model):
    return model.calibrated_classifiers_[0].estimator if isinstance(model, CalibratedClassifierCV) else model


def export_interpretation(run, max_patients=300):
    bundle, dev, out = run["bundle"], run["development"], run["output_dir"]
    cfg = bundle["frozen"]["config"]
    features = bundle["frozen"]["selected_features"]
    lr = base_pipeline(bundle["models"]["logistic_regression"])
    names = lr.named_steps["preprocess"].get_feature_names_out()
    coefficients = pd.DataFrame({"feature": names, "coefficient": lr.named_steps["model"].coef_[0]})
    coefficients["abs_coefficient"] = coefficients.coefficient.abs()
    coefficients.sort_values("abs_coefficient", ascending=False).to_csv(out / "logistic_base_coefficients.csv", index=False)
    # Native TreeSHAP contributions sum to the base model's log odds.
    xgb = base_pipeline(bundle["models"]["xgboost"])
    sample = dev[features].sample(min(max_patients, len(dev)), random_state=cfg["seed"])
    transformed = xgb.named_steps["preprocess"].transform(sample)
    booster = xgb.named_steps["model"].model_.get_booster()
    contrib = booster.predict(DMatrix(transformed), pred_contribs=True)
    margin = booster.predict(DMatrix(transformed), output_margin=True)
    np.testing.assert_allclose(contrib.sum(axis=1), margin, atol=1e-4, rtol=1e-4)
    shap_table = pd.DataFrame({"feature": xgb.named_steps["preprocess"].get_feature_names_out(),
        "mean_abs_shap_log_odds": np.abs(contrib[:, :-1]).mean(axis=0)}).sort_values("mean_abs_shap_log_odds", ascending=False)
    shap_table.to_csv(out / "xgboost_base_treeshap.csv", index=False)
    top = shap_table.head(15).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top.feature, top.mean_abs_shap_log_odds)
    ax.set(xlabel="Mean absolute TreeSHAP contribution (base-model log odds)", title="Development sample: predictive associations, not causal effects")
    fig.tight_layout()
    fig.savefig(out / "xgboost_base_treeshap.png", dpi=180, bbox_inches="tight")
    plt.close(fig)
    return coefficients, shap_table

In [ ]:
for analysis, run in runs.items():
    print(analysis)
    lr_coefficients, xgb_shap = export_interpretation(run)
    display(lr_coefficients.sort_values("abs_coefficient", ascending=False).head(10))
    display(Image(filename=str(run["output_dir"] / "xgboost_base_treeshap.png")))

## 8. Final evaluation

After trial checks and full development, freeze every analysis and enable `run_final_test` in main mode. A suite-wide lock records all data, settings and model hashes before the first holdout is scored. Identical reruns reuse the frozen models/results; changed analyses are rejected after final evaluation starts.

Final outputs include metrics and curves, patient bootstrap 95% intervals, paired XGBoost–LR differences, error summaries and exploratory sex/gender results. With the ablation enabled, paired bootstrap differences compare six-hour versus last-hour models on identical test patients. Intervals are conditional on the fitted models; small event counts and subgroups may be unstable. Interpret filtered/full native-test comparisons descriptively because their populations differ.

Synthetic check mode exercises this stage automatically. Trial mode cannot evaluate real holdouts.

In [ ]:
def bootstrap_intervals(p, thresholds, cfg):
    """Use the same resampled patients for every model in each bootstrap replicate."""
    rng = np.random.default_rng(cfg["seed"] + 200)
    samples, differences = [], []
    for rep in range(cfg["bootstrap_replicates"]):
        sample = p.iloc[rng.integers(0, len(p), len(p))]
        if sample.y_true.nunique() < 2:
            continue
        table = comparison_tables(sample, thresholds)
        for _, row in table.iterrows():
            for metric in ["auroc", "average_precision", "sensitivity", "specificity", "ppv", "npv", "brier"]:
                samples.append({"replicate": rep, "scope": row.scope, "model": row.model, "metric": metric, "value": row.get(metric, np.nan)})
        paired = table[table.scope.eq("all_patients")].set_index("model")
        for metric in ["auroc", "average_precision", "brier"]:
            differences.append({"replicate": rep, "metric": metric, "value": paired.loc["xgboost", metric] - paired.loc["logistic_regression", metric]})
    if not samples:
        raise ValueError("No usable bootstrap replicates; insufficient test events.")
    def summarise(frame, groups):
        return frame.groupby(groups).value.agg(valid_replicates="count", lower_95=lambda s: s.quantile(.025), upper_95=lambda s: s.quantile(.975)).reset_index()
    return summarise(pd.DataFrame(samples), ["scope", "model", "metric"]), summarise(pd.DataFrame(differences), ["metric"])


def final_evaluation(run, enabled=False):
    if not enabled:
        print("Final test disabled. Enable it after fixing models, features and thresholds.")
        return None
    bundle, df, out = run["bundle"], run["data"], run["output_dir"]
    frozen = bundle["frozen"]
    cfg = frozen["config"]
    synthetic_check = cfg["run_mode"] == "check" and cfg["synthetic_data"]
    if cfg["run_mode"] != "main" and not synthetic_check:
        raise ValueError("Final evaluation is disabled for trial runs. Complete a main run first.")
    marker = out / "final_test_started.json"
    signature = hashlib.sha256(json.dumps(frozen, sort_keys=True).encode()).hexdigest()
    model_hash = file_hash(out / "trained_models.joblib")
    if marker.exists():
        prior = json.loads(marker.read_text())
        if prior.get("analysis_signature") != signature or prior.get("model_sha256") != model_hash:
            raise RuntimeError("Models or settings changed after final evaluation started.")
        if prior["status"] == "completed":
            print("Loading saved final-test results.")
            return pd.read_csv(out / "FINAL_test_metrics.csv")
        print("Resuming final evaluation with the saved models.")
    # Save the marker before evaluation so interrupted runs can resume.
    marker_info = {"analysis_signature": signature, "model_sha256": model_hash,
        "selected_primary_model": frozen["selected_primary_model"], "synthetic_data": cfg["synthetic_data"]}
    write_json(marker, {**marker_info, "status": "started"})
    test = df.set_index("subject_id", drop=False).loc[bundle["test_ids"]].reset_index(drop=True)
    X = test[frozen["selected_features"]]
    scores = {n: m.predict_proba(X)[:, 1] for n, m in bundle["models"].items()}
    scores["prevalence_baseline"] = np.full(len(test), bundle["development_prevalence"])
    scores["random_guess"] = np.random.default_rng(cfg["seed"] + 300).uniform(size=len(test))
    p = prediction_table(test, cfg, scores, frozen["rsbi_column"])
    p.to_csv(out / "FINAL_test_predictions.csv", index=False)
    table = comparison_tables(p, frozen["thresholds"])
    table.to_csv(out / "FINAL_test_metrics.csv", index=False)
    plot_comparison(p, out, "FINAL_test")
    if "rsbi" in p:
        plot_comparison(p.loc[np.isfinite(p.rsbi)], out, "FINAL_test_rsbi_matched_subset", include_rsbi=True)
    if p.y_true.nunique() == 2 and cfg["bootstrap_replicates"] > 0:
        ci, differences = bootstrap_intervals(p, frozen["thresholds"], cfg)
        ci.to_csv(out / "FINAL_test_patient_bootstrap_CI.csv", index=False)
        differences.to_csv(out / "FINAL_paired_difference_XGB_minus_LR_CI.csv", index=False)
    else:
        warnings.warn("Test set has one outcome class or bootstrap is disabled; AUROC or intervals may be unavailable.")
    # Apply the same frozen thresholds to error and subgroup summaries.
    for name in MODEL_NAMES:
        p[name + "_predicted_failure"] = (p[name] >= frozen["thresholds"][name]).astype(int)
        p[name + "_error"] = np.select([(p.y_true == 1) & (p[name + "_predicted_failure"] == 0),
            (p.y_true == 0) & (p[name + "_predicted_failure"] == 1)], ["false_negative", "false_positive"], default="correct")
    p.to_csv(out / "FINAL_test_error_analysis.csv", index=False)
    error_summary = []
    age_col = next((c for c in ["admission_age", "age"] if c in p), None)
    for name in MODEL_NAMES:
        for error, group in p.groupby(name + "_error"):
            row = {"model": name, "error": error, "patients": len(group),
                "death_events": int(group["death_within_48_hours"].sum()) if "death_within_48_hours" in group else np.nan,
                "reintubation_events": int(group["reintubation_within_48_hours"].sum()) if "reintubation_within_48_hours" in group else np.nan}
            if age_col:
                row["median_age"] = group[age_col].median()
            error_summary.append(row)
    pd.DataFrame(error_summary).to_csv(out / "FINAL_error_summary.csv", index=False)
    write_json(out / "FINAL_followup.json", followup_summary(test, cfg))
    subgroup_rows = []
    for col in ["gender", "sex"]:
        if col in p:
            for value, group in p.groupby(col, dropna=False):
                for name in MODEL_NAMES:
                    subgroup_rows.append({"variable": col, "group": str(value), "model": name,
                        "interpretation": "exploratory; small subgroups may be unstable", **metrics(group.y_true, group[name], frozen["thresholds"][name])})
    if subgroup_rows:
        pd.DataFrame(subgroup_rows).to_csv(out / "FINAL_exploratory_subgroup_metrics.csv", index=False)
    write_json(marker, {**marker_info, "status": "completed",
        "CI_interpretation": "Conditional on fitted models and this test cohort; does not include uncertainty from repeating model development."})
    return table


def paired_ablation_comparison(runs):
    """Compare feature sets on identical test patients, separately for LR and XGBoost."""
    main = runs["filtered_six_hour"]
    ablation = runs["filtered_last_hour"]
    a = pd.read_csv(main["output_dir"] / "FINAL_test_predictions.csv", dtype={"subject_id": "string"}).set_index("subject_id").sort_index()
    b = pd.read_csv(ablation["output_dir"] / "FINAL_test_predictions.csv", dtype={"subject_id": "string"}).set_index("subject_id").sort_index()
    pd.testing.assert_series_equal(a.y_true, b.y_true)
    cfg = main["bundle"]["frozen"]["config"]
    rng = np.random.default_rng(cfg["seed"] + 400)
    keys = ["auroc", "average_precision", "brier"]
    point = {name: (metrics(a.y_true, a[name]), metrics(b.y_true, b[name])) for name in MODEL_NAMES}
    differences = {(name, metric): [] for name in MODEL_NAMES for metric in keys}
    for _ in range(cfg["bootstrap_replicates"]):
        idx = rng.integers(0, len(a), len(a))
        y = a.y_true.iloc[idx]
        if y.nunique() < 2:
            continue
        for name in MODEL_NAMES:
            ma, mb = metrics(y, a[name].iloc[idx]), metrics(y, b[name].iloc[idx])
            for metric in keys:
                differences[name, metric].append(ma[metric] - mb[metric])
    rows = []
    for name in MODEL_NAMES:
        ma, mb = point[name]
        for metric in keys:
            values = differences[name, metric]
            rows.append({"model": name, "metric": metric, "test_patients": len(a),
                "six_hour": ma[metric], "last_hour": mb[metric],
                "six_hour_minus_last_hour": ma[metric] - mb[metric],
                "lower_95": np.quantile(values, .025) if values else np.nan,
                "upper_95": np.quantile(values, .975) if values else np.nan,
                "valid_replicates": len(values),
                "interpretation": "paired patients; conditional on fitted models; lower Brier is better"})
    table = pd.DataFrame(rows)
    table.to_csv(Path(cfg["suite_output_dir"]) / "FINAL_paired_feature_ablation.csv", index=False)
    return table


def evaluate_suite(runs, configs, enabled=False):
    if not enabled:
        print("Development complete for all analyses. Real holdouts remain unscored.")
        return None
    for cfg in configs.values():
        if cfg["run_mode"] != "main" and not (cfg["run_mode"] == "check" and cfg["synthetic_data"]):
            raise ValueError("Final evaluation is unavailable in trial mode.")
    verify_ablation_pair(runs)
    guard_final_suite(configs)
    folder = Path(next(iter(configs.values()))["suite_output_dir"])
    marker = folder / "final_suite_started.json"
    if not marker.exists():
        for name, cfg in configs.items():
            frozen = runs[name]["bundle"]["frozen"]
            if cfg != frozen["config"] or load_cohort(cfg)[-1] != frozen["source_hashes"]:
                raise RuntimeError("Data or settings changed since development. Refit all analyses before opening the holdout.")
        write_json(marker, {"status": "started", "primary_analysis": "filtered_six_hour",
            "plan": suite_plan(configs),
            "model_hashes": {name: file_hash(run["output_dir"] / "trained_models.joblib") for name, run in runs.items()},
            "analysis_hashes": {name: file_hash(run["output_dir"] / "frozen_analysis.json") for name, run in runs.items()}})
    # The complete suite is locked BEFORE any overlapping holdout is scored.
    for name, run in runs.items():
        print("Final evaluation:", name)
        final_evaluation(run, enabled=True)
    result = export_analysis_comparison(runs, final=True)
    if "filtered_last_hour" in runs:
        paired_ablation_comparison(runs)
    saved = json.loads(marker.read_text())
    write_json(marker, {**saved, "status": "completed"})
    return result

In [ ]:
final_results = evaluate_suite(
    runs, RUN_CONFIGS,
    enabled=CONFIG["execution"]["mode"] == "check" or CONFIG["execution"]["run_final_test"],
)
if final_results is not None:
    display(final_results)
    for analysis, run in runs.items():
        print(analysis)
        display(Image(filename=str(run["output_dir"] / "FINAL_test_curves.png")))
    if "filtered_last_hour" in runs:
        display(pd.read_csv(Path(RUN_CONFIGS["filtered_six_hour"]["suite_output_dir"]) / "FINAL_paired_feature_ablation.csv"))
if CONFIG["execution"]["mode"] == "check":
    for run in runs.values():
        assert run["oof"].subject_id.is_unique
        assert all(run["oof"][name].between(0, 1).all() for name in PROBABILITY_MODELS)
        assert not set(run["bundle"]["development_ids"]) & set(run["bundle"]["test_ids"])
        assert (run["output_dir"] / "FINAL_paired_difference_XGB_minus_LR_CI.csv").exists()
    print("Synthetic pipeline check passed. These results must not be reported as patient findings.")